# BirdCLEF+ 2026 - Ensemble Inference (Submission)

**Environment:** Kaggle CPU notebook (no GPU, no internet, 90-min limit)

This notebook loads multiple fold models and ensembles their predictions.
Budget: ~600 files x 12 windows = ~7200 predictions within 90 minutes on CPU.

**Required Kaggle Datasets:**
1. `birdclef-2026` - competition data
2. `birdclef-model-weights` - all fold .pth files
3. `birdclef-src` - source code

In [ ]:
import os
import sys
import time
import glob

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

START_TIME = time.time()

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    KAGGLE_USER = "stochastix"
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
    WEIGHTS_DIR = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-model-weights"
    SRC_DIR = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src"
    sys.path.insert(0, SRC_DIR)
else:
    DATA_DIR = "../data"
    WEIGHTS_DIR = "../models"
    sys.path.insert(0, "..")

TEST_DIR = os.path.join(DATA_DIR, "test_soundscapes")
TAXONOMY_CSV = os.path.join(DATA_DIR, "taxonomy.csv")
SAMPLE_SUB = os.path.join(DATA_DIR, "sample_submission.csv")

print(f"ON_KAGGLE: {ON_KAGGLE}")
print(f"TEST_DIR: {TEST_DIR}")
print(f"WEIGHTS_DIR: {WEIGHTS_DIR}")

In [ ]:
import yaml
from src.audio import load_audio, audio_to_melspec, normalize_melspec
from src.transforms import spectrogram_to_tensor
from src.dataset import load_taxonomy
from src.model import get_model
from src.utils import Timer, ensemble_predictions

# Load species list
species_list = load_taxonomy(TAXONOMY_CSV)
print(f"Species: {len(species_list)}")

# Runtime timer
timer = Timer(budget_seconds=5400)  # 90 minutes
print(timer)

## Load Models

In [ ]:
# Find all model weight files
weight_files = sorted(glob.glob(os.path.join(WEIGHTS_DIR, "*_best.pth")))
print(f"Found {len(weight_files)} model weight files:")
for wf in weight_files:
    print(f"  {os.path.basename(wf)}")

# Load models
models = []
device = torch.device("cpu")

for wf in weight_files:
    checkpoint = torch.load(wf, map_location=device)
    config = checkpoint.get("config", None)
    
    if config is None:
        # Fallback: load config from file
        config_path = os.path.join(SRC_DIR, "config", "sed_efficientnet_b1.yaml") if ON_KAGGLE else "../config/sed_efficientnet_b1.yaml"
        with open(config_path) as f:
            config = yaml.safe_load(f)
    
    model = get_model(config["model"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    models.append((model, config))
    
    fold = checkpoint.get("fold", "?")
    auc = checkpoint.get("val_auc", 0)
    print(f"  Loaded fold {fold}: val_auc={auc:.4f}")

print(f"\n{len(models)} models loaded")
print(timer)

## Enumerate Test Windows

In [ ]:
import librosa

test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(".ogg")])
print(f"Test soundscapes: {len(test_files)}")

# Pre-compute all windows
windows = []
for filename in tqdm(test_files, desc="Indexing windows"):
    file_path = os.path.join(TEST_DIR, filename)
    total_dur = librosa.get_duration(path=file_path)
    basename = os.path.splitext(filename)[0]

    start = 0.0
    duration = 5.0
    while start + duration <= total_dur + 0.01:
        end = start + duration
        row_id = f"{basename}_{int(end)}"
        windows.append({
            "file_path": file_path,
            "start": start,
            "row_id": row_id,
        })
        start += duration

print(f"Total windows: {len(windows)}")
print(timer)

## Run Ensemble Inference

In [ ]:
# Use config from first model
audio_config = models[0][1]["audio"]
SR = audio_config["sample_rate"]
DURATION = audio_config["duration"]
IN_CHANNELS = models[0][1]["model"].get("in_channels", 1)
BATCH_SIZE = 32  # CPU batch size

# Collect predictions per model
all_model_preds = [[] for _ in models]  # one list per model
all_row_ids = []

for batch_start in tqdm(range(0, len(windows), BATCH_SIZE), desc="Inference"):
    # Check time budget
    timer.check(required_seconds=120, msg="Stopping early to save submission")
    
    batch_windows = windows[batch_start:batch_start + BATCH_SIZE]
    batch_tensors = []
    batch_row_ids = []

    for w in batch_windows:
        audio = load_audio(w["file_path"], sr=SR, duration=DURATION, offset=w["start"])
        melspec = audio_to_melspec(audio, SR, audio_config)
        melspec = normalize_melspec(melspec)
        tensor = spectrogram_to_tensor(melspec, IN_CHANNELS)
        batch_tensors.append(tensor)
        batch_row_ids.append(w["row_id"])

    batch_input = torch.stack(batch_tensors)
    all_row_ids.extend(batch_row_ids)

    # Run each model
    for model_idx, (model, config) in enumerate(models):
        with torch.no_grad():
            output = model(batch_input)
            if isinstance(output, dict):
                logits = output["clipwise_logits"]
            else:
                logits = output
            probs = torch.sigmoid(logits).numpy()
            all_model_preds[model_idx].append(probs)

# Stack predictions per model
stacked_preds = [np.concatenate(preds, axis=0) for preds in all_model_preds]

print(f"\nPredictions shape per model: {stacked_preds[0].shape}")
print(f"Models: {len(stacked_preds)}")
print(timer)

In [ ]:
# Ensemble: simple mean averaging
# For more sophisticated ensembling, try 'geometric' or 'rank' or weighted by val_auc
final_preds = ensemble_predictions(stacked_preds, method="mean")

print(f"Final predictions shape: {final_preds.shape}")
print(f"Prediction range: [{final_preds.min():.6f}, {final_preds.max():.6f}]")
print(f"Mean prediction: {final_preds.mean():.6f}")

## Create Submission

In [ ]:
# Build submission DataFrame
rows = []
for row_id, probs in zip(all_row_ids, final_preds):
    row = {"row_id": row_id}
    for species, prob in zip(species_list, probs):
        row[species] = float(prob)
    rows.append(row)

submission = pd.DataFrame(rows)

# Ensure column order matches sample submission
if os.path.exists(SAMPLE_SUB):
    sample = pd.read_csv(SAMPLE_SUB, nrows=1)
    submission = submission[sample.columns]

# Save
submission.to_csv("submission.csv", index=False)
print(f"Submission shape: {submission.shape}")
submission.head()

## Sanity Checks

In [ ]:
# Validate submission
assert submission.shape[1] == len(species_list) + 1, f"Expected {len(species_list)+1} cols, got {submission.shape[1]}"
assert not submission.isnull().any().any(), "Submission contains NaN!"

prob_cols = submission.columns[1:]
prob_values = submission[prob_cols].values
assert prob_values.min() >= 0, f"Min prob: {prob_values.min()}"
assert prob_values.max() <= 1, f"Max prob: {prob_values.max()}"

# Distribution stats
print("All checks passed!")
print(f"Rows: {len(submission)}")
print(f"Prob range: [{prob_values.min():.6f}, {prob_values.max():.6f}]")
print(f"Mean prob: {prob_values.mean():.6f}")
print(f"Median prob: {np.median(prob_values):.6f}")

# Per-species activity
active_per_species = (prob_values > 0.5).sum(axis=0)
print(f"\nSpecies with >0.5 predictions: {(active_per_species > 0).sum()}/{len(species_list)}")
print(f"\nTotal runtime: {time.time() - START_TIME:.1f}s ({(time.time() - START_TIME)/60:.1f} min)")
print(f"Budget remaining: {90 - (time.time() - START_TIME)/60:.1f} min")